In [15]:
import numpy as np
import pandas as pd
from PIL import Image
from pathlib import Path
from tqdm import tqdm
from sklearn.feature_extraction.image import extract_patches_2d
from sklearn.preprocessing import StandardScaler
import joblib
from matplotlib.colors import rgb_to_hsv
from scipy.ndimage import sobel

In [16]:
BASE_DIR = Path(".")
CSV_PATH = BASE_DIR / "dataset" / "dataset_crawl_070526_with_images_path.csv"
BIN_DIR = BASE_DIR / "bin"
BIN_DIR.mkdir(parents=True, exist_ok=True)

In [17]:
df = pd.read_csv(CSV_PATH)
mask = df["Screenshot_Path"].notna() & (df["Screenshot_Path"] != "")
df_img = df[mask].copy().reset_index(drop=True)
print(f"Images: {len(df_img)}")

Images: 37811


In [18]:
PATCH_SIZE = (16, 16)
MAX_PATCHES = 50
RESIZE = (64, 64)
N_HIST_BINS = 32

In [19]:
def extract_features(img_path):
    img = Image.open(img_path).convert("RGB")
    img_resized = img.resize(RESIZE, Image.LANCZOS)
    arr = np.array(img_resized, dtype=np.uint8)

    # 1. Patch features (sklearn)
    patches = extract_patches_2d(arr, patch_size=PATCH_SIZE, max_patches=MAX_PATCHES)
    patch_feat = []
    for p in patches:
        for c in range(3):
            patch_feat.append(p[:, :, c].mean())
            patch_feat.append(p[:, :, c].std())

    # 2. RGB histogram
    hist_feat = []
    for c in range(3):
        h = img.histogram()[c * 256 : (c + 1) * 256]
        bin_sz = 256 // N_HIST_BINS
        binned = [sum(h[j * bin_sz : (j + 1) * bin_sz]) for j in range(N_HIST_BINS)]
        total = sum(binned) + 1e-8
        hist_feat.extend(b / total for b in binned)

    # 3. HSV histogram
    arr_float = arr / 255.0
    hsv = rgb_to_hsv(arr_float)
    h_ch, s_ch, _ = hsv[:, :, 0], hsv[:, :, 1], hsv[:, :, 2]
    h_hist, _ = np.histogram(h_ch, bins=N_HIST_BINS, range=(0, 360), density=True)
    s_hist, _ = np.histogram(s_ch, bins=N_HIST_BINS, range=(0, 1), density=True)
    hsv_feat = np.concatenate([h_hist, s_hist])

    # 4. Edge density (Sobel via scipy)
    gray = np.array(img_resized.convert("L"), dtype=np.float64)
    edges_x = sobel(gray, axis=1)
    edges_y = sobel(gray, axis=0)
    edges_mag = np.hypot(edges_x, edges_y)
    max_mag = edges_mag.max()
    if max_mag > 0:
        edge_feat = np.array([
            edges_mag.mean() / max_mag,
            edges_mag.std() / max_mag,
            (edges_mag > edges_mag.mean()).sum() / edges_mag.size,
        ])
    else:
        edge_feat = np.zeros(3)

    # 5. Warm color ratio (H in 0-60 or 330-360)
    warm_mask = (h_ch >= 0) & (h_ch <= 60) | (h_ch >= 330) & (h_ch <= 360)
    warm_ratio = np.array([warm_mask.sum() / h_ch.size])

    # 6. Color variance across 4x4 grid
    h_cells, w_cells = 4, 4
    cell_h, cell_w = arr.shape[0] // h_cells, arr.shape[1] // w_cells
    cell_means = np.zeros((h_cells * w_cells, 3))
    idx = 0
    for i in range(h_cells):
        for j in range(w_cells):
            cell = arr[i * cell_h : (i + 1) * cell_h, j * cell_w : (j + 1) * cell_w]
            cell_means[idx] = cell.mean(axis=(0, 1))
            idx += 1
    color_var = cell_means.std(axis=0)

    return np.concatenate([patch_feat, hist_feat, hsv_feat, edge_feat, warm_ratio, color_var])

In [20]:
feats, ids = [], []
for _, row in tqdm(df_img.iterrows(), total=len(df_img)):
    try:
        feats.append(extract_features(row["Screenshot_Path"].replace("model/", "")))
        ids.append(row["Webpage_id"])
    except Exception as e:
        print(f"Skip {row['Screenshot_Path']}: {e}")

100%|██████████| 37811/37811 [15:39<00:00, 40.25it/s]


In [21]:
X = np.array(feats)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [22]:
np.save(BIN_DIR / "image_features.npy", X_scaled)
np.save(BIN_DIR / "image_ids.npy", np.array(ids))
joblib.dump(scaler, BIN_DIR / "image_scaler.pkl")

['bin/image_scaler.pkl']

In [23]:
PATCH_FEAT_DIM = MAX_PATCHES * 3 * 2
HIST_FEAT_DIM = 3 * N_HIST_BINS
HSV_FEAT_DIM = 2 * N_HIST_BINS
EDGE_FEAT_DIM = 3
WARM_DIM = 1
COLOR_VAR_DIM = 3
TOTAL_DIM = PATCH_FEAT_DIM + HIST_FEAT_DIM + HSV_FEAT_DIM + EDGE_FEAT_DIM + WARM_DIM + COLOR_VAR_DIM

In [24]:
print(f"Feature dimension: {X_scaled.shape[1]} (expected {TOTAL_DIM})")
dims = dict(patch=PATCH_FEAT_DIM, rgb_hist=HIST_FEAT_DIM, hsv_hist=HSV_FEAT_DIM, edge=EDGE_FEAT_DIM, warm=WARM_DIM, color_var=COLOR_VAR_DIM)
print(f"Breakdown: {dims}")
print(f"Saved: image_features.npy ({X_scaled.shape})")

Feature dimension: 467 (expected 467)
Breakdown: {'patch': 300, 'rgb_hist': 96, 'hsv_hist': 64, 'edge': 3, 'warm': 1, 'color_var': 3}
Saved: image_features.npy ((37811, 467))
